# Task #50 — Đánh giá 3 mô hình trên tập test (Story #9)

Đọc `orders_features_test.csv` (chưa dùng đến từ Task #46), load 3 model đã huấn luyện ở Task #49 (`models/*.pkl`), dự đoán trên tập test và tính F1/Precision/Recall cho từng model. Kết hợp với thời gian huấn luyện đã đo ở Task #49 (`models/training_times.json`) thành 1 bảng so sánh duy nhất, làm căn cứ chọn mô hình để tinh chỉnh ở Story #10.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)

df["is_delayed"] = df["is_delayed"].astype(bool)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

X_test = df.drop(columns=["order_id", "is_delayed"])
y_test = df["is_delayed"].astype(int)

print("X_test:", X_test.shape, " y_test:", y_test.shape)

X_test: (19289, 75)  y_test: (19289,)


## 1. Load 3 model đã huấn luyện + thời gian huấn luyện (Task #49)

Không huấn luyện lại — dùng nguyên `models/*.pkl` và `models/training_times.json` đã có sẵn.

In [2]:
models_dir = Path("../models")

log_reg = joblib.load(models_dir / "logistic_regression.pkl")
rand_forest = joblib.load(models_dir / "random_forest.pkl")
xgb = joblib.load(models_dir / "xgboost.pkl")

with open(models_dir / "training_times.json", encoding="utf-8") as f:
    training_times = json.load(f)

models = {
    "logistic_regression": log_reg,
    "random_forest": rand_forest,
    "xgboost": xgb,
}

print(training_times)

{'logistic_regression': 31.861822000006214, 'random_forest': 3.7170604999992065, 'xgboost': 3.0953199000068707}


## 2. Dự đoán trên tập test, tính F1/Precision/Recall

Lớp dương (`pos_label=1`) là đơn **trễ** (`is_delayed=True`) — khớp định nghĩa bài toán ở README (F1 mục tiêu ≥ 0.78 đo trên lớp trễ, không phải trung bình 2 lớp).

In [3]:
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        "model": name,
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1": f1_score(y_test, y_pred, pos_label=1),
        "training_time_s": training_times[name],
    })

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df

,model,precision,recall,f1,training_time_s
0,xgboost,0.211338,0.595527,0.311967,3.095320
1,random_forest,0.411279,0.191054,0.260908,3.717060
2,logistic_regression,0.118558,0.655591,0.200802,31.861822


## 3. Lưu bảng so sánh

Lưu vào `models/evaluation_results.csv` (commit git — không phải file `.pkl` nặng) để Story #10 tham chiếu khi chọn mô hình tinh chỉnh.

In [4]:
results_df.to_csv(models_dir / "evaluation_results.csv", index=False)
print("Da luu bang so sanh vao models/evaluation_results.csv")
results_df

Da luu bang so sanh vao models/evaluation_results.csv


,model,precision,recall,f1,training_time_s
0,xgboost,0.211338,0.595527,0.311967,3.095320
1,random_forest,0.411279,0.191054,0.260908,3.717060
2,logistic_regression,0.118558,0.655591,0.200802,31.861822
